# Marketing P&L Simulator - Example Analysis

This notebook demonstrates how to use the Marketing P&L Simulator to analyze scenarios.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.models.pl_calculator import PLCalculator
from src.data.scenario import Scenario
from src.data.loader import DataLoader
import pandas as pd

## Load Example Scenarios


In [ ]:
scenarios = DataLoader.load_scenarios_from_csv('../data/scenarios/example_scenarios.csv')
for scenario in scenarios:
    print(f'{scenario.name}: ${scenario.revenue:,}')

## Calculate P&L for Each Scenario


In [ ]:
calculator = PLCalculator()
results = [calculator.calculate(s) for s in scenarios]

# Display results as DataFrame
results_data = [r.to_dict() for r in results]
df_results = pd.DataFrame(results_data)
df_results

## Compare Against Base Case


In [ ]:
comparison = calculator.compare(results, base_index=0)

print('Incremental Analysis vs Base Case:\n')
for inc in comparison['incremental_analysis']:
    print(f"{inc['vs']}")
    print(f"  Contribution Delta: ${inc['contribution_delta']:,.0f} ({inc['contribution_delta_pct']:.1f}%)")
    print()

## Create Custom Scenarios


In [ ]:
# Define custom scenarios
custom_scenarios = [
    Scenario(name='Base', revenue=1000000, cogs_percent=0.45, marketing_spend=50000, trade_spend=30000),
    Scenario(name='Aggressive Marketing', revenue=1050000, cogs_percent=0.45, marketing_spend=100000, trade_spend=40000),
]

custom_results = [calculator.calculate(s) for s in custom_scenarios]
custom_comparison = calculator.compare(custom_results)

df_custom = pd.DataFrame([r.to_dict() for r in custom_results])
df_custom

## Sensitivity Analysis


In [ ]:
# Test sensitivity to marketing spend
base = Scenario(name='Base', revenue=1000000, cogs_percent=0.45, marketing_spend=50000, trade_spend=30000)

marketing_sensitivity = []
for spend in [30000, 50000, 75000, 100000, 150000]:
    scenario = Scenario(
        name=f'Marketing ${spend:,}',
        revenue=1000000,
        cogs_percent=0.45,
        marketing_spend=spend,
        trade_spend=30000
    )
    result = calculator.calculate(scenario)
    marketing_sensitivity.append({
        'Marketing Spend': f'${spend:,}',
        'Contribution': f'${result.contribution:,}',
        'Contribution %': f'{result.contribution_pct:.1%}'
    })

pd.DataFrame(marketing_sensitivity)

## Export Results


In [ ]:
# Save results to CSV
results_for_export = [r.to_dict() for r in results]
DataLoader.save_results_to_csv(results_for_export, '../data/processed/scenario_results.csv')
print('Results exported to data/processed/scenario_results.csv')

---

**Next Steps:**
- Modify scenarios to test different hypotheses
- Add volume lift or price change assumptions
- Extend model with additional cost categories
